# 08b -- WalterFootball Rankings

**Purpose**: Scrape WalterFootball position-specific 2026 draft prospect rankings
and ingest into `fact_rookie_rankings`. Self-contained.

**Sources scraped**: 13 position pages (QB, RB, WR, TE, DE, DT, OLB3-4, DE3-4, NT, OLB, ILB, CB, S)

**Aggregation**: After scraping, rows are grouped by `(player, scraped position_group)` and
`positional_rank` is averaged. A defender ranked on multiple scheme-specific pages within the
same position group (e.g., DT + NT both → DL) produces **one** ingested row with the averaged rank.
A player on pages from *different* groups (e.g., DE → DL, OLB34 → LB) produces **two** rows.

**Ingested source_names**: `WalterFootball_{position_group}` — e.g., `WalterFootball_DL`,
`WalterFootball_LB`, `WalterFootball_DB`, `WalterFootball_RB`, etc.

**`global_rank`**: null — WalterFootball does not publish a cross-position big board.

**`rank_date`**: extracted from "This page was last updated Month DD, YYYY" in page HTML.

**Prerequisites**: `08_fact_rookie_rankings_seed.ipynb`, `01_dim_rookie_prospect.ipynb`

**Outputs**: `data/fact_rookie_rankings.parquet` (appended), `data/review_fuzzy_matches.csv` (if needed)

## 1 -- Setup & Shared Helpers

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import requests
import re
import json
import time
from pathlib import Path
from datetime import date, datetime
from dataclasses import dataclass, field
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from thefuzz import fuzz, process


@dataclass
class LeagueConfig:
    draft_year: int = 2026
    total_cap: int = 500_000_000
    num_teams: int = 28
    num_conferences: int = 2
    initial_contract_years: int = 3
    extension_contract_years: int = 3
    fa_minimum_salary: int = 2_000_000
    data_dir: str = "data"
    review_dir: str = "data/review"          # all review CSVs live here
    fuzzy_auto_threshold: int = 90
    fuzzy_review_threshold: int = 70


CFG    = LeagueConfig()
DATA   = Path(CFG.data_dir)
REVIEW = Path(CFG.review_dir)
TODAY  = date.today().isoformat()
DATA.mkdir(exist_ok=True)
REVIEW.mkdir(parents=True, exist_ok=True)
ALIAS  = DATA / "dim_player_alias.parquet"   # persistent name->player_key decisions (08y/08z)


def _make_session(timeout_sec: int = 30, retries: int = 3, backoff: float = 2.0) -> requests.Session:
    # Shared session factory: retry on timeout/5xx with exponential backoff.
    # backoff waits: 2s, 4s, 8s between attempts.
    session = requests.Session()
    retry = Retry(
        total=retries,
        backoff_factor=backoff,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://",  adapter)
    return session

In [2]:
# -- Player name/key helpers (copied from 01_dim_rookie_prospect) -------------

def clean_player_name(name: str) -> str:
    # Normalize for matching: remove periods, collapse whitespace, lowercase.
    if pd.isna(name):
        return ""
    s = str(name).strip()
    s = s.replace(".", "").replace("\u00a0", " ")
    s = s.replace("\u2018", "'").replace("\u2019", "'").replace("`", "'")
    return " ".join(s.split()).lower()


def generate_player_key(name: str, position: str, school: str) -> str:
    # Deterministic 12-char MD5 hash -- matches keys generated in 01.
    raw = f"{clean_player_name(name)}|{str(position).upper().strip()}|{str(school).strip().lower()}"
    return hashlib.md5(raw.encode()).hexdigest()[:12]


assert clean_player_name("J.K. Dobbins") == "jk dobbins"
assert clean_player_name("D'Andre Swift") == "d'andre swift"
print("Helpers OK")

def _parse_rank_date(raw: str | None) -> str | None:
    # Parse the source-published "last updated" date into ISO format.
    # Returns None if raw is empty or unparseable (caller stores NULL).
    if not raw:
        return None
    raw = str(raw).strip()
    for fmt in ("%B %d, %Y", "%b %d, %Y", "%Y-%m-%d", "%m/%d/%Y"):
        try:
            return datetime.strptime(raw, fmt).date().isoformat()
        except ValueError:
            continue
    return raw  # store as-is if no format matched


Helpers OK


## 2 -- add_players_from_source

In [ ]:
def add_players_from_source(
    new_players_df: pd.DataFrame,
    source_name: str,
    draft_year: int = CFG.draft_year,
    auto_threshold: int = CFG.fuzzy_auto_threshold,
    review_threshold: int = CFG.fuzzy_review_threshold,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Fuzzy-match new player names against dim_rookie_prospect.
    # Returns (updated_dim_rookie_prospect, review_df).
    # new_players_df must have: player_name; optionally position_raw, school_raw.
    # NOTE: review_df is returned to caller -- file write is the caller's responsibility.
    dim_rp      = pd.read_parquet(DATA / "dim_rookie_prospect.parquet")
    pos_map     = pd.read_parquet(DATA / "dim_position.parquet")
    school_map  = pd.read_parquet(DATA / "dim_school.parquet")

    existing_names  = dim_rp["player_name_clean"].tolist()
    existing_lookup = dict(zip(dim_rp["player_name_clean"], dim_rp["player_key"]))

    # Persistent decisions: (name_clean, position_raw) already resolved in a prior
    # review -> never ask again (dim_player_alias, built by 08y, appended by 08z).
    alias_keys = set()
    if ALIAS.exists():
        _al = pd.read_parquet(ALIAS)
        alias_keys = set(zip(_al["name_clean"], _al["position_raw"]))

    new_rows, review_rows = [], []
    alias_rows = []
    auto_matched = already_exists = alias_resolved = 0

    for _, row in new_players_df.iterrows():
        name_clean = clean_player_name(row["player_name"])
        pos_key    = str(row.get("position_raw", "")).upper().strip()

        if name_clean in existing_lookup:
            already_exists += 1
            continue

        # Already decided in a past review -> skip silently (no repeat review).
        if (name_clean, pos_key) in alias_keys:
            alias_resolved += 1
            continue

        best_match, score = ("", 0)
        if existing_names:
            best_match, score = process.extractOne(
                name_clean, existing_names, scorer=fuzz.token_sort_ratio
            )

        if score >= auto_threshold:
            auto_matched += 1
            # Link the new name variant so ingest_ranking_source can find it.
            matched_key = existing_lookup[best_match]
            existing_lookup[name_clean] = matched_key
            existing_names.append(name_clean)
            base = dim_rp[dim_rp["player_name_clean"] == best_match].iloc[0].to_dict()
            alias_rows.append({**base, "player_name_clean": name_clean, "player_name": row["player_name"]})
        elif score >= review_threshold:
            review_rows.append({
                "new_name":        row["player_name"],
                "new_name_clean":  name_clean,
                "new_position":    row.get("position_raw", ""),
                "new_school":      row.get("school_raw", ""),
                "best_match_name": best_match,
                "best_match_key":  existing_lookup.get(best_match, ""),
                "fuzzy_score":     score,
                "action":          "",  # fill: "match" or "new"
                "source":          source_name,
            })
        else:
            # New prospect -- add to dim_rookie_prospect
            pos_raw    = str(row.get("position_raw", "")).upper().strip()
            school_raw = str(row.get("school_raw", "")).strip()
            pos_match  = pos_map[pos_map["position_raw"] == pos_raw]
            sch_match  = school_map[school_map["school_raw"] == school_raw]
            pkey       = generate_player_key(row["player_name"], pos_raw, school_raw)
            new_rows.append({
                "player_key":       pkey,
                "player_name":      row["player_name"],
                "player_name_clean": name_clean,
                "position_raw":     pos_raw,
                "position_detail":  pos_match["position_detail"].iloc[0] if len(pos_match) else None,
                "position_group":   pos_match["position_group"].iloc[0]  if len(pos_match) else None,
                "side_of_ball":     pos_match["side_of_ball"].iloc[0]    if len(pos_match) else None,
                "fantasy_relevant": pos_match["fantasy_relevant"].iloc[0] if len(pos_match) else False,
                "school_raw":       school_raw,
                "school_canonical": sch_match["school_canonical"].iloc[0] if len(sch_match) else school_raw,
                "conference":       sch_match["conference"].iloc[0]       if len(sch_match) else "Unknown",
                "height_inches":    None, "weight": None,
                "pfr_id":           None, "cfb_id": None,
                "gsis_id":          pd.NA,
                "draft_year":       draft_year,
                "source":           source_name,
                "added_date":       TODAY,
            })
            existing_names.append(name_clean)
            existing_lookup[name_clean] = pkey

    if new_rows or alias_rows:
        dim_rp = pd.concat([dim_rp, pd.DataFrame(new_rows + alias_rows)], ignore_index=True)
        dim_rp.drop_duplicates(subset=["player_name_clean"], inplace=True)
        dim_rp.to_parquet(DATA / "dim_rookie_prospect.parquet", index=False)

    review_df = pd.DataFrame(review_rows) if review_rows else pd.DataFrame()

    print(f"Source: {source_name}")
    print(f"  Already in dim_rookie_prospect : {already_exists}")
    print(f"  Resolved via alias (no re-ask)  : {alias_resolved}")
    print(f"  Auto-matched (>={auto_threshold})       : {auto_matched}")
    print(f"  New prospects added             : {len(new_rows)}")
    print(f"  Needs manual review             : {len(review_rows)}")

    return dim_rp, review_df

## 3 -- ingest_ranking_source

In [ ]:
def ingest_ranking_source(
    rankings_df: pd.DataFrame,
    source_name: str,
    source_site: str,
    phase: str,
    draft_year: int = CFG.draft_year,
    capture_date: str = TODAY,
    rank_date: str | None = None,
) -> pd.DataFrame:
    # Append one row per player to fact_rookie_rankings for a given source + phase.
    # Call add_players_from_source() first to ensure all players are in dim_rookie_prospect.
    # Players pending review (not yet in dim_rookie_prospect) are logged as unmatched and skipped.
    #
    # rankings_df required columns: player_name, global_rank
    # rankings_df optional columns: positional_rank, grade

    valid_phases = {"pre_combine", "post_combine", "post_draft"}
    if phase not in valid_phases:
        raise ValueError(f"phase must be one of {valid_phases}")

    # name -> player_key lookup from dim_rookie_prospect
    dim_rp = pd.read_parquet(DATA / "dim_rookie_prospect.parquet")
    name_to_key = dict(zip(dim_rp["player_name_clean"], dim_rp["player_key"]))
    key_to_pfr  = dict(zip(dim_rp["player_key"],        dim_rp["pfr_id"]))

    # Fold in alias decisions so a matched name-variant (X resolved to prospect Y)
    # attributes its ranking to Y's player_key instead of being dropped as unmatched.
    # setdefault: real dim_rookie_prospect names always win over an alias.
    if ALIAS.exists():
        _al = pd.read_parquet(ALIAS)
        for _nc, _pk in zip(_al["name_clean"], _al["player_key"]):
            name_to_key.setdefault(_nc, _pk)

    # pfr_id -> gsis_id lookup from dim_nfl_players (null pre-signing)
    dim_nfl = pd.read_parquet(DATA / "dim_nfl_players.parquet")
    pfr_to_gsis = dict(zip(dim_nfl["pfr_id"].dropna(), dim_nfl["gsis_id"].dropna()))

    rows, unmatched = [], []
    for _, row in rankings_df.iterrows():
        name_clean = clean_player_name(row["player_name"])
        pkey = name_to_key.get(name_clean)

        if pkey is None:
            unmatched.append(row["player_name"])
            continue

        pfr_id  = key_to_pfr.get(pkey)
        gsis_id = pfr_to_gsis.get(pfr_id) if pfr_id else None

        rows.append({
            "player_key":      pkey,
            "gsis_id":         gsis_id,
            "source_name":     source_name,
            "source_site":     source_site,
            "phase":           phase,
            "draft_year":      draft_year,
            "global_rank":     row.get("global_rank"),
            "positional_rank": row.get("positional_rank"),
            "grade":           row.get("grade"),
            "capture_date":    capture_date,
            "rank_date":       rank_date,
        })

    if unmatched:
        print(f"WARN: {len(unmatched)} players pending review -- will be picked up after apply_review_decisions():")
        for name in unmatched[:10]:
            print(f"  {name}")
        if len(unmatched) > 10:
            print(f"  ... and {len(unmatched) - 10} more")

    new_df = pd.DataFrame(rows)
    if new_df.empty:
        print("No rows to append.")
        return new_df

    new_df["global_rank"]     = new_df["global_rank"].astype("Int64")
    new_df["positional_rank"] = new_df["positional_rank"].astype("Int64")
    new_df["draft_year"]      = new_df["draft_year"].astype("Int64")
    new_df["rank_date"]       = new_df["rank_date"].where(new_df["rank_date"].notna(), other=None)

    existing = pd.read_parquet(DATA / "fact_rookie_rankings.parquet")
    combined = pd.concat([existing, new_df], ignore_index=True)
    _DEDUP   = ["player_key", "source_name", "phase", "draft_year"]
    # rank_date: preserve the first non-null value ever captured — never overwrite with a later scrape
    combined["rank_date"] = (
        combined.groupby(_DEDUP, sort=False)["rank_date"]
        .transform(lambda s: s.dropna().iloc[0] if s.notna().any() else None)
    )
    combined.drop_duplicates(subset=_DEDUP, keep="last", inplace=True)
    combined.to_parquet(DATA / "fact_rookie_rankings.parquet", index=False)

    print(f"Ingested: {len(rows)} rows | source={source_name} | site={source_site} | phase={phase}")
    print(f"fact_rookie_rankings total: {len(combined)}")
    return new_df

## 4 -- WalterFootballScraper

Players are in `<b>` tags inside `<article>`, in ranked order (positional rank =
position in document). Format: `"Name*, POS, SchoolHeight: X-X. Weight: XXX."`

In [5]:
class WalterFootballScraper:
    # Scrapes position-specific 2026 draft prospect rankings from WalterFootball.com.
    # Players are in <b> tags inside <article>, in ranked order (rank = doc position).
    # Format: "Name*, POS, SchoolHeight: X-X. Weight: XXX. Projected Round (YYYY): X-X."

    _HEADERS = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }

    _PLAYER_PAT = re.compile(
        r"^([A-Za-z][A-Za-z\s'\-.]+?)\*?"
        r",\s+([A-Z0-9\-]{1,8})"
        r",\s+(.+?)(?=Height:|$)",
        re.DOTALL,
    )
    _ROUND_PAT   = re.compile(r"Projected Round \(\d{4}\):\s*([\d\-]+)")
    _UPDATED_PAT = re.compile(r"last updated\s+([A-Za-z]+\s+\d{1,2},\s*\d{4})", re.IGNORECASE)
    _POS_NORM  = {"3-4DE": "DE", "3-4OLB": "OLB", "4-3DE": "DE"}
    _SKIP      = {"Previous", "Walt", "Charlie", "Season", "Grade"}

    SOURCES = {
        "WalterFootball_QB":    {"url": "https://walterfootball.com/QBRankings2026.php",     "site_reference": "2026 NFL Draft Prospect Rankings: Quarterbacks",        "source_site": "WalterFootball"},
        "WalterFootball_RB":    {"url": "https://walterfootball.com/RBRankings2026.php",     "site_reference": "2026 NFL Draft Prospect Rankings: Running Backs",        "source_site": "WalterFootball"},
        "WalterFootball_WR":    {"url": "https://walterfootball.com/WRRankings2026.php",     "site_reference": "2026 NFL Draft Prospect Rankings: Wide Receivers",       "source_site": "WalterFootball"},
        "WalterFootball_TE":    {"url": "https://walterfootball.com/TERankings2026.php",     "site_reference": "2026 NFL Draft Prospect Rankings: Tight Ends",           "source_site": "WalterFootball"},
        "WalterFootball_DE":    {"url": "https://walterfootball.com/DERankings2026.php",     "site_reference": "2026 NFL Draft Prospect Rankings: Defensive Ends",       "source_site": "WalterFootball"},
        "WalterFootball_DT":    {"url": "https://walterfootball.com/DTRankings2026.php",     "site_reference": "2026 NFL Draft Prospect Rankings: Defensive Tackles",    "source_site": "WalterFootball"},
        "WalterFootball_OLB34": {"url": "https://walterfootball.com/OLB3-4Rankings2026.php","site_reference": "2026 NFL Draft Prospect Rankings: 3-4 Rush Linebackers", "source_site": "WalterFootball"},
        "WalterFootball_DE34":  {"url": "https://walterfootball.com/DE3-4Rankings2026.php", "site_reference": "2026 NFL Draft Prospect Rankings: 3-4 Defensive Ends",   "source_site": "WalterFootball"},
        "WalterFootball_NT":    {"url": "https://walterfootball.com/NTRankings2026.php",     "site_reference": "2026 NFL Draft Prospect Rankings: Nose Tackles",         "source_site": "WalterFootball"},
        "WalterFootball_OLB":   {"url": "https://walterfootball.com/OLBRankings2026.php",   "site_reference": "2026 NFL Draft Prospect Rankings: Outside Linebackers",  "source_site": "WalterFootball"},
        "WalterFootball_ILB":   {"url": "https://walterfootball.com/ILBRankings2026.php",   "site_reference": "2026 NFL Draft Prospect Rankings: Inside Linebackers",   "source_site": "WalterFootball"},
        "WalterFootball_CB":    {"url": "https://walterfootball.com/CBRankings2026.php",     "site_reference": "2026 NFL Draft Prospect Rankings: Cornerbacks",          "source_site": "WalterFootball"},
        "WalterFootball_S":     {"url": "https://walterfootball.com/SRankings2026.php",     "site_reference": "2026 NFL Draft Prospect Rankings: Safeties",             "source_site": "WalterFootball"},
    }

    def __init__(self, timeout: int = 30, delay: float = 2.0):
        self.timeout = timeout
        self.delay   = delay
        self.session = _make_session(timeout_sec=timeout)
        self.session.headers.update(self._HEADERS)

    def scrape(self, source_key: str) -> pd.DataFrame:
        cfg = self.SOURCES[source_key]
        time.sleep(self.delay)
        resp = self.session.get(cfg["url"], timeout=self.timeout)
        resp.raise_for_status()

        from bs4 import BeautifulSoup
        soup    = BeautifulSoup(resp.text, "html.parser")
        article = soup.find("article")
        if not article:
            raise ValueError(f"No <article> at {cfg['url']}")

        # Parse source-published date from "This page was last updated Month DD, YYYY"
        m_date = self._UPDATED_PAT.search(soup.get_text())
        rank_date = _parse_rank_date(m_date.group(1) if m_date else None)

        rows = []
        for b_tag in article.find_all("b"):
            text = b_tag.get_text(separator="", strip=True)
            m    = self._PLAYER_PAT.match(text)
            if not m:
                continue
            name = m.group(1).strip()
            if any(skip in name for skip in self._SKIP):
                continue
            pos    = self._POS_NORM.get(m.group(2).strip().upper(), m.group(2).strip().upper())
            school = m.group(3).strip()
            rnd_m  = self._ROUND_PAT.search(text)
            rows.append({
                "player_name":     name,
                "position_raw":    pos,
                "school_raw":      school,
                "global_rank":     None,
                "positional_rank": len(rows) + 1,
                "grade":           None,
                "proj_round":      rnd_m.group(1) if rnd_m else None,
            })

        if not rows:
            raise ValueError(f"No player entries parsed from {cfg['url']}")

        df = pd.DataFrame(rows)
        print(f"Scraped {source_key}: {len(df)} players | {cfg['site_reference']} | rank_date={rank_date}")
        return df, rank_date

## 5 -- Run WalterFootball Ingestion

In [6]:
from bs4 import BeautifulSoup   # pip install beautifulsoup4

WF_SCRAPER   = WalterFootballScraper(timeout=30, delay=2.0)
PHASE        = "post_draft"
_failed      = []
_all_reviews = []
_scraped     = {}  # source_key -> (df, rank_date)

# ── Step 1: Scrape all 13 position pages ─────────────────────────────────────
for source_key in WalterFootballScraper.SOURCES:
    print("\n" + "=" * 60)
    print(f"Scraping: {source_key}")
    try:
        rankings_df, rank_date = WF_SCRAPER.scrape(source_key)
        _scraped[source_key] = (rankings_df, rank_date)
    except Exception as e:
        print(f"  ERROR: {e}")
        _failed.append(source_key)

# ── Step 2: add_players_from_source per page (updates dim_rookie_prospect) ────
print("\n" + "=" * 60)
print("Fuzzy-matching players into dim_rookie_prospect...")
for source_key, (rankings_df, _) in _scraped.items():
    try:
        _dim_rp, review = add_players_from_source(
            rankings_df, source_name=source_key, draft_year=CFG.draft_year
        )
        if not review.empty:
            print(f"  -> {len(review)} players flagged for review ({source_key})")
            _all_reviews.append(review)
    except Exception as e:
        print(f"  ERROR add_players ({source_key}): {e}")
        _failed.append(source_key)

# ── Step 3: Aggregate by scraped position_group (avg positional_rank) ─────────
# Players ranked on multiple pages within the same position_group get their
# positional_rank averaged — one row per player per group, no arbitrary winner.
# Players spanning different groups (e.g., DE->DL and OLB->LB) keep both rows.
print("\n" + "=" * 60)
print("Aggregating by position_group...")

_all_dfs = []
for source_key, (df, rank_date) in _scraped.items():
    _df = df.copy()
    _df["_wf_source"]   = source_key
    _df["_rank_date"]  = rank_date
    _all_dfs.append(_df)

combined_wf = pd.concat(_all_dfs, ignore_index=True)

# Join scraped position_raw -> position_group from dim_position
pos_map = pd.read_parquet(DATA / "dim_position.parquet")[["position_raw", "position_group"]]
combined_wf = combined_wf.merge(pos_map, on="position_raw", how="left")
combined_wf["_name_clean"] = combined_wf["player_name"].apply(clean_player_name)

agg_wf = (
    combined_wf.groupby(["_name_clean", "position_group"], dropna=False)
    .agg(
        player_name    = ("player_name",    "first"),
        position_raw   = ("position_raw",   "first"),
        school_raw     = ("school_raw",     "first"),
        global_rank    = ("global_rank",    "first"),   # always null for WF
        positional_rank= ("positional_rank","mean"),
        grade          = ("grade",          "first"),   # always null for WF
        rank_date      = ("_rank_date",     "max"),     # latest date across contributing pages
        _sources       = ("_wf_source",    lambda x: "+".join(sorted(set(x)))),
    )
    .reset_index()
)
agg_wf["positional_rank"] = agg_wf["positional_rank"].round().astype("Int64")

pre  = len(combined_wf)
post = len(agg_wf)
print(f"  {pre} scraped rows -> {post} aggregated rows ({pre - post} merged by avg)")

averaged = agg_wf[agg_wf["_sources"].str.contains(r"\+", regex=True)]
if len(averaged):
    print(f"  {len(averaged)} players averaged across pages within same position_group:")
    for _, r in averaged.iterrows():
        print(f"    {r['player_name']:30s}  {r['position_group']}  "
              f"<- {r['_sources']}  (avg rank={r['positional_rank']})")

# ── Step 4: Ingest per position_group ─────────────────────────────────────────
print("\n" + "=" * 60)
print("Ingesting aggregated WalterFootball rows...")
SOURCES = {}  # populated here; used by validation cell

_nan_rows = agg_wf[agg_wf["position_group"].isna()]
if len(_nan_rows):
    print(f"WARN: {len(_nan_rows)} rows have no position_group mapping (position_raw not in dim_position) -- skipped:")
    for _, _r in _nan_rows.iterrows():
        print(f"  {_r['player_name']}  ({_r['position_raw']})")

for pos_group, group_df in agg_wf.groupby("position_group", dropna=False):
    if pd.isna(pos_group):
        continue  # already reported in WARN block above
    source_key = f"WalterFootball_{pos_group}"
    rd         = group_df["rank_date"].max()
    SOURCES[source_key] = {"source_site": "WalterFootball"}
    try:
        ingest_ranking_source(
            group_df.reset_index(drop=True),
            source_name=source_key,
            source_site="WalterFootball",
            phase=PHASE,
            draft_year=CFG.draft_year,
            rank_date=rd,
        )
    except Exception as e:
        print(f"  ERROR ingesting {source_key}: {e}")
        _failed.append(source_key)

# ── Write review file (append-safe across notebooks) ─────────────────────────
if _all_reviews:
    new_review = pd.concat(_all_reviews, ignore_index=True)
    rp = DATA / "review" / "review_fuzzy_matches.csv"
    if rp.exists():
        existing = pd.read_csv(rp)
        combined = pd.concat([existing, new_review], ignore_index=True)
        # keep="first" preserves already-filled action values from prior review sessions
        combined.drop_duplicates(subset=["new_name", "source"], keep="first", inplace=True)
        n_added = len(combined) - len(existing)
        combined.to_csv(rp, index=False)
        print("\n" + "=" * 60)
        print(f"Review file updated: +{max(n_added, 0)} new rows ({len(combined)} total) -> {rp}")
    else:
        new_review.to_csv(rp, index=False)
        print("\n" + "=" * 60)
        print(f"Review file: {rp}  ({len(new_review)} rows)")
    print("Fill \"action\" column (\"match\" or \"new\"), run 09_apply_fuzzy_review, then re-run.")

print("\n" + "=" * 60)
if _failed:
    print(f"Failed: {_failed}")
else:
    print("All WalterFootball sources completed.")


Scraping: WalterFootball_QB
Scraped WalterFootball_QB: 17 players | 2026 NFL Draft Prospect Rankings: Quarterbacks | rank_date=2026-04-19

Scraping: WalterFootball_RB
Scraped WalterFootball_RB: 17 players | 2026 NFL Draft Prospect Rankings: Running Backs | rank_date=2026-04-19

Scraping: WalterFootball_WR
Scraped WalterFootball_WR: 43 players | 2026 NFL Draft Prospect Rankings: Wide Receivers | rank_date=2026-04-19

Scraping: WalterFootball_TE
Scraped WalterFootball_TE: 22 players | 2026 NFL Draft Prospect Rankings: Tight Ends | rank_date=2026-04-19

Scraping: WalterFootball_DE
Scraped WalterFootball_DE: 27 players | 2026 NFL Draft Prospect Rankings: Defensive Ends | rank_date=2026-04-19

Scraping: WalterFootball_DT
Scraped WalterFootball_DT: 24 players | 2026 NFL Draft Prospect Rankings: Defensive Tackles | rank_date=2026-04-19

Scraping: WalterFootball_OLB34
Scraped WalterFootball_OLB34: 26 players | 2026 NFL Draft Prospect Rankings: 3-4 Rush Linebackers | rank_date=2026-04-19

Scra

## 6 -- Validation

In [7]:
# Mini-validation: rows written by this source
import pandas as pd
from pathlib import Path
fr = pd.read_parquet(Path(CFG.data_dir) / "fact_rookie_rankings.parquet")
src_keys = list(SOURCES.keys())
sub = fr[fr["source_name"].isin(src_keys)]
print(f"Rows for this source: {len(sub)}")
print(sub.groupby(["source_name", "phase"]).size().to_string())
print()
print(f"fact_rookie_rankings total: {len(fr)} rows")
dupes = fr.duplicated(subset=["player_key","source_name","phase","draft_year"]).sum()
print(f"Composite key duplicates: {dupes} (expected 0)")

Rows for this source: 209
source_name        phase     
WalterFootball_DB  post_draft    42
WalterFootball_DL  post_draft    46
WalterFootball_LB  post_draft    28
WalterFootball_QB  post_draft    17
WalterFootball_RB  post_draft    16
WalterFootball_TE  post_draft    20
WalterFootball_WR  post_draft    40

fact_rookie_rankings total: 572 rows
Composite key duplicates: 0 (expected 0)
